# 05 Feedback Loop: TTRL

**Scenario**: Test-Time Reinforcement Learning (TTRL, Zuo et al. 2025) trains a model on **unlabeled** test questions: sample many candidate answers per question, treat the majority-vote answer as a pseudo-label, reward answers that agree with the majority, update the model, and repeat. It is a closed loop: "generate → score → update → generate again."

Normally this requires two separate systems—vLLM for sampling and a separate training framework for updates, syncing weights from trainer to inference engine every round. In pLLM, sampling and training share a single engine and a single set of weights: the workload declares a LoRA and its training method, while a plain Python loop on the host runs majority voting, packs training batches, and feeds them back into the next `generate` call.

This tutorial implements the simplest form of TTRL: a supervised update step on samples matching the majority answer (equivalent to policy gradient with 0/1 rewards and no baseline). We run a few rounds on 16 GSM8K questions and watch majority accuracy and inter-sample agreement evolve. Finally, we swap the same loop to GRPO: keeping the same reward, but weighting each sample by its within-group advantage.

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # Show only vLLM errors to suppress startup logs

import json
import math
import re
import time
from collections import Counter
from dataclasses import dataclass, replace

import torch
from transformers import AutoTokenizer
from vllm import SamplingParams

import pllm
from pllm.host import CausalTrainBatch

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

## Step 1: Workload—One LoRA and One Training Method

- `lora` is an engine-wide `pllm.InstanceState` shared across requests, with low-rank factor fields `a` and `b`. `pllm.Field`'s shape function receives each target projection's input and output widths, so one declaration allocates a pair of factors for `q` and `v` at every layer.
- `adapter` applies `lora` across every layer's `q` and `v` projections: `y = W x + scale · B A x`. `lora_init` initializes `b` to zero so the model starts identical to the base checkpoint, while initializing `a` randomly—if both factors start at zero, every gradient is zero and the LoRA never learns. pLLM therefore requires optimizer-backed low-rank adapters to provide nonzero initialization (`init=pllm.Computed(...)`).
- `learn` is decorated with `@pllm.train(state="lora")`. It reads a host batch (`pllm.batch(...)`) plus `pllm.causal_model("adapter")`: a full causal LM carrying the current LoRA where gradients flow only into the LoRA parameters. `model.token_logprobs(ids, labels, ...)` returns the log-probability of the next label token at each position (and `0` where `labels == -100`), chunking the LM head so the full batch's logits never need to materialize at once. The method returns a scalar loss; the framework handles backward and AdamW.

In [2]:
def lora_init(fields: dict[str, torch.Tensor], generator: torch.Generator) -> None:
    for name, value in fields.items():
        if name.startswith("a_"):
            torch.nn.init.kaiming_uniform_(value, a=math.sqrt(5), generator=generator)
        else:
            value.zero_()


@dataclass(frozen=True)
class TTRL(pllm.Workload):
    rank: int = 8
    learning_rate: float = 1e-4

    @property
    def lora(self) -> pllm.InstanceState:
        return pllm.InstanceState(
            {"a": pllm.Field(lambda inputs, outputs: (self.rank, inputs)),
             "b": pllm.Field(lambda inputs, outputs: (outputs, self.rank))},
            optimizer=pllm.AdamW(self.learning_rate), init=pllm.Computed(lora_init))

    @property
    def adapter(self) -> pllm.model.Adapter:
        targets = (pllm.model.Projection("q_proj"), pllm.model.Projection("v_proj"))  # Every layer
        return pllm.model.Adapter("lora", targets, operator="low_rank", scale=2.0)

    @pllm.train(state="lora")
    def learn(self, input_ids: torch.Tensor = pllm.batch("input_ids"), labels: torch.Tensor = pllm.batch("labels"),
              attention_mask: torch.Tensor = pllm.batch("attention_mask"),
              model: pllm.model.CausalModel = pllm.causal_model("adapter")) -> torch.Tensor:
        logprobs = model.token_logprobs(input_ids, labels, attention_mask=attention_mask)  # [batch, seq_len - 1]
        return -logprobs.sum() / labels[:, 1:].ne(-100).sum()  # Mean negative log-prob over answer tokens

In `lora_init`, field names look like `a_0, b_0, a_1, ...`: one slice per target projection, numbered in model order.

## Step 2: Majority Voting on the Host

This part is plain Python: extract answers from `\boxed{}`, tally the majority vote, select samples matching the majority, and pack them into a `CausalTrainBatch` (setting prompt label positions to `-100` so loss is computed only on answer tokens). To bound training VRAM per step, each round selects only the 8 questions with the strongest majority consensus, taking one matching sample per question.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
with open("gsm8k_test_16.json") as file:  # First 16 questions from GSM8K test split
    tests = json.load(file)


def normalize(text: str) -> str | None:
    try:
        return f"{float(text.replace(',', '').replace('$', '').strip()):g}"
    except ValueError:
        return None


def boxed(text: str) -> str | None:
    found = re.findall(r"\\boxed\{([^{}]*)\}", text)
    return normalize(found[-1]) if found else None


gold = [normalize(row["a"]) for row in tests]
prompts = [tokenizer.apply_chat_template(
    [{"role": "user", "content": row["q"] + "\nPut the final answer in \\boxed{}."}],
    tokenize=False, add_generation_prompt=True) for row in tests]


def vote(outputs) -> tuple[float, float, list[tuple[list[int], list[int]]]]:
    # Returns: majority-vote accuracy, mean majority vote share, and (prompt, answer) token sequences for training.
    correct, agreement, candidates = 0, 0.0, []
    for prompt, output, answer in zip(prompts, outputs, gold):
        answers = [boxed(sample.text) for sample in output.outputs]
        counts = Counter(a for a in answers if a is not None)
        if not counts:
            continue
        majority, count = counts.most_common(1)[0]
        correct += majority == answer
        agreement += count / len(answers)
        sample = next(s for s, a in zip(output.outputs, answers) if a == majority)
        candidates.append((count, tokenizer(prompt, add_special_tokens=False).input_ids, list(sample.token_ids)))
    # Top 8 questions with highest majority agreement, one matching sample each.
    candidates.sort(key=lambda item: -item[0])
    return correct / len(prompts), agreement / len(prompts), [(p, a) for _, p, a in candidates[:8]]


def training_batch(sequences: list[tuple[list[int], list[int]]]) -> CausalTrainBatch:
    width = max(len(p) + len(a) for p, a in sequences)
    ids = torch.full((len(sequences), width), tokenizer.pad_token_id)
    labels = torch.full_like(ids, -100)
    mask = torch.zeros_like(ids)
    for row, (p, a) in enumerate(sequences):
        ids[row, :len(p) + len(a)] = torch.tensor(p + a)
        labels[row, len(p):len(p) + len(a)] = torch.tensor(a)
        mask[row, :len(p) + len(a)] = 1
    return CausalTrainBatch(ids.cuda(), labels.cuda(), mask.cuda())

## Step 3: Closing the Loop

Each round samples 8 completions for each of the 16 questions (128 concurrent requests), runs majority voting, and passes the selected samples from the previous round via `batch=` into the next `generate` call: the framework runs one AdamW step at the start of that call, and the updated LoRA takes effect immediately for subsequent tokens.
`max_host_training_rows` sets the maximum sequence count accepted per training step and must be declared upfront in `ExecutionOptions` so the framework can budget GPU memory.

In [4]:
# Give vLLM only 25% GPU memory, leaving the rest for full-model forward/backward during training.
config = pllm.ModelConfig(MODEL, max_model_len=640, max_num_seqs=128, max_num_batched_tokens=2048,
                          gpu_memory_utilization=0.25, seed=0,
                          execution=pllm.ExecutionOptions(max_host_training_rows=8))
sampling = SamplingParams(n=8, temperature=0.8, top_p=0.95, max_tokens=256, seed=0)

with pllm.make_llm(config, workload=TTRL()) as llm:
    batch = None
    for round_index in range(5):
        start = time.perf_counter()
        outputs = llm.generate(prompts, sampling, batch=batch)
        accuracy, agreement, chosen = vote(outputs)
        print(f"第 {round_index} 轮：多数答案准确率 {accuracy:.2f}，多数票占比 {agreement:.2f}，"
              f"挑出 {len(chosen)} 个样本，用时 {time.perf_counter() - start:.0f}s，"
              f"已更新 {llm.diagnostics().update_count} 次")
        batch = training_batch(chosen)

第 0 轮：多数答案准确率 0.31，多数票占比 0.16，挑出 8 个样本，用时 16s，已更新 0 次


第 1 轮：多数答案准确率 0.25，多数票占比 0.16，挑出 8 个样本，用时 54s，已更新 1 次


第 2 轮：多数答案准确率 0.38，多数票占比 0.19，挑出 8 个样本，用时 43s，已更新 2 次


第 3 轮：多数答案准确率 0.38，多数票占比 0.20，挑出 8 个样本，用时 11s，已更新 3 次


第 4 轮：多数答案准确率 0.38，多数票占比 0.24，挑出 8 个样本，用时 44s，已更新 4 次


Across five rounds, majority accuracy fluctuates between 0.25 and 0.38 while majority agreement rises from 0.16 to 0.24. On just 16 questions that is only a one- or two-question difference and **not evidence of algorithmic efficacy**: this tutorial demonstrates how to wire the feedback loop, not a reproduction of TTRL results. Agreement is low mainly because this 0.5B model frequently omits `\boxed{}` formatting, and responses without a parsed answer count as 0.

Look at the elapsed times as well: some rounds are noticeably slower (40–55s vs. 11s) because pLLM recompiles the training graph whenever the host training batch takes on a new shape.

## Step 4: Switching to GRPO

TTRL uses only samples matching the majority, all with weight `1`. GRPO (Shao et al. 2024) scores each question's group of samples jointly: subtract the group mean reward and divide by the group standard deviation to get each sample's **advantage**; positive-advantage samples are reinforced while negative-advantage samples are suppressed. The reward remains "matches the majority answer."

Only two changes compared to TTRL:

- The training method reads an extra `pllm.batch("weights")`: one scalar weight per sequence—its advantage. The loss computes each sample's mean negative log-probability over answer tokens, weighted by advantage and averaged;
- The host computes advantages and passes them in `CausalTrainBatch.weights`.

Because `max_host_training_rows=8` fits all 8 samples of one question, each round picks the question with the highest non-unanimous majority count (if all 8 agree, rewards are identical and advantages are all zero, yielding zero signal). Here we perform a single online step without importance-ratio clipping or a KL penalty.

In [5]:
@dataclass(frozen=True)
class GRPO(TTRL):
    @pllm.train(state="lora")
    def learn(self, input_ids: torch.Tensor = pllm.batch("input_ids"), labels: torch.Tensor = pllm.batch("labels"),
              attention_mask: torch.Tensor = pllm.batch("attention_mask"),
              weights: torch.Tensor = pllm.batch("weights"),
              model: pllm.model.CausalModel = pllm.causal_model("adapter")) -> torch.Tensor:
        logprobs = model.token_logprobs(input_ids, labels, attention_mask=attention_mask)
        per_sample = logprobs.sum(-1) / labels[:, 1:].ne(-100).sum(-1)  # Mean log-prob over answer tokens per sample
        return -(weights * per_sample).mean()


def grpo_group(outputs) -> CausalTrainBatch:
    """Pick the question with highest non-unanimous majority count: all its samples weighted by within-group normalized advantage."""
    groups = []
    for prompt, output in zip(prompts, outputs):
        answers = [boxed(sample.text) for sample in output.outputs]
        counts = Counter(a for a in answers if a is not None)
        if not counts:
            continue
        majority, count = counts.most_common(1)[0]
        if count == len(answers):  # Unanimous: identical rewards in the group give zero signal
            continue
        rewards = torch.tensor([float(a == majority) for a in answers])
        ids = tokenizer(prompt, add_special_tokens=False).input_ids
        groups.append((count, [(ids, list(sample.token_ids)) for sample in output.outputs],
                       (rewards - rewards.mean()) / rewards.std(correction=0)))
    _, sequences, advantages = max(groups, key=lambda group: group[0])
    return replace(training_batch(sequences), weights=advantages.cuda())

In [6]:
with pllm.make_llm(config, workload=GRPO()) as llm:
    batch = None
    for round_index in range(5):
        start = time.perf_counter()
        outputs = llm.generate(prompts, sampling, batch=batch)
        accuracy, agreement, _ = vote(outputs)
        print(f"第 {round_index} 轮：多数答案准确率 {accuracy:.2f}，多数票占比 {agreement:.2f}，"
              f"用时 {time.perf_counter() - start:.0f}s，已更新 {llm.diagnostics().update_count} 次")
        batch = grpo_group(outputs)

第 0 轮：多数答案准确率 0.31，多数票占比 0.16，用时 11s，已更新 0 次


第 1 轮：多数答案准确率 0.38，多数票占比 0.16，用时 43s，已更新 1 次


第 2 轮：多数答案准确率 0.38，多数票占比 0.17，用时 12s，已更新 2 次


第 3 轮：多数答案准确率 0.31，多数票占比 0.17，用时 44s，已更新 3 次


第 4 轮：多数答案准确率 0.38，多数票占比 0.16，用时 46s，已更新 4 次


Across the five GRPO rounds, majority accuracy stays between 0.31 and 0.38 and majority agreement between 0.16 and 0.17. Updating on just 8 samples from a single question per round, as in the TTRL section, only shows the loop executing end-to-end rather than benchmark performance.

## Summary

- Host-batch training methods read a host batch (`pllm.batch(...)`) and the LoRA-equipped causal model (`pllm.causal_model(...)`), returning a scalar loss;
- Voting, scoring, and sample selection stay in plain host Python, fed back into the engine via the next `generate(batch=...)`;
- Advantage-weighted updates (GRPO) simply add a per-sequence `weights` field to the batch and multiply it in the loss;
- Sampling and training share one engine and one set of weights—no weight synchronization needed.

pLLM also provides a higher-level host feedback helper: `llm.generate_feedback(...)` invokes your feedback callback after each round, retries selected requests according to its decision, and manages voting groups. We didn't use it here because this loop regenerates all questions every round and feeds training batches into the LoRA via `generate(batch=...)`, whereas `generate_feedback` retries only flagged requests and passes `StateImport` between rounds. The built-in `pllm.workloads.RewardHeadFeedback` is one such callback: it runs majority voting on `\boxed{}` answers and fits a reward head (`pllm.workloads.RewardHeadWorkload`) to those 0/1 rewards to predict reward from each response's final hidden state. A reward head trains purely on the host without running the base model: `pllm.host.make_host(config, workload)` returns a `HostTrainer` whose `train(batch)` runs one step.

## Common Mistakes

Two common mistakes in host-batch training: passing `reduction=` out of habit from per-row training, and leaving both LoRA factors zero-initialized.

In [7]:
@dataclass(frozen=True)
class SummedTTRL(TTRL):
    @pllm.train(state="lora", reduction="sum")  # Host-batch loss is already a scalar
    def learn(self, input_ids: torch.Tensor = pllm.batch("input_ids"), labels: torch.Tensor = pllm.batch("labels"),
              attention_mask: torch.Tensor = pllm.batch("attention_mask"),
              model: pllm.model.CausalModel = pllm.causal_model("adapter")) -> torch.Tensor:
        return -model.token_logprobs(input_ids, labels, attention_mask=attention_mask).sum()


@dataclass(frozen=True)
class ZeroStart(TTRL):
    @property
    def lora(self) -> pllm.InstanceState:  # No init=: both factors start at zero
        return pllm.InstanceState({"a": pllm.Field(lambda inputs, outputs: (self.rank, inputs)),
                                   "b": pllm.Field(lambda inputs, outputs: (outputs, self.rank))},
                                  optimizer=pllm.AdamW(self.learning_rate))


CPU_CONFIG = pllm.ModelConfig(MODEL, max_num_seqs=8)
for workload in (SummedTTRL(), ZeroStart()):
    try:
        pllm.explain(workload, CPU_CONFIG)
    except ValueError as error:
        print(f"{type(workload).__name__}: {error}\n")

SummedTTRL: host-batch training returns one scalar and owns its reduction; remove reduction=

ZeroStart: lora: add init=pllm.Computed(...) (e.g. random A, zero B); with both low_rank factors at zero every gradient is zero and the adapter never learns



**Why host-batch training rejects `reduction=`.** In per-row training, the scheduler determines how many rows belong to each state copy in a step, so pLLM must perform the reduction. A host batch consists of complete sequences you assembled yourself—only you know sequence lengths, weights, and whether to average per token or per sequence (precisely what distinguishes GRPO, DAPO, and Dr.GRPO)—so the method returns a scalar directly and owns its reduction.

**Why zero-initialized LoRA never learns.** In `B A x`, the gradient with respect to `A` contains `B`, and the gradient with respect to `B` contains `A`. When both are zero, both gradients are identically zero and the optimizer can never move them away from zero. The standard initialization is random `A` and zero `B`: zero delta before training starts, nonzero gradients on step 1.

## Self-Check

1. Why must `max_host_training_rows` be declared upfront in `ExecutionOptions`?

<details><summary>Answer</summary>

Full-model training needs substantial temporary GPU memory. pLLM reserves that memory upfront at engine startup according to your declared maximum batch size instead of gambling on dynamic allocations mid-generation; batches exceeding the declared row limit are rejected.
</details>

2. Why were a few rounds much slower than others?

<details><summary>Answer</summary>

Whenever the training batch changes shape (number of sequences or sequence length), pLLM recompiles the training step for the new shape. Once shapes stabilize, recompilation stops.
</details>

3. What changed in the workload and host code when switching from TTRL to GRPO?

<details><summary>Answer</summary>

The workload's training method added one `pllm.batch("weights")` input and multiplied each sample's loss by its weight; the host code computed within-group advantages and passed them in `CausalTrainBatch.weights`. The engine, LoRA, and loop structure stayed unchanged.
</details>

This concludes the core tutorial track. From here, explore as needed: [Workload Anatomy](../guides/workload-anatomy.md) systematically organizes every pattern introduced across these six notebooks; [Core Concepts](../explain/core-concepts.md) explains the design behind the rules; and the [Examples](../README.md#examples) cover one algorithm family per notebook, such as [06 Reading Inside](../examples/06_reading_inside.ipynb) for reading the residual stream and per-layer logits to implement DoLa.